In [1]:
import os, time
print("PID:", os.getpid())
print("Time:", time.time())

PID: 9396
Time: 1782717905.2785034


In [2]:
import pandas as pd
df = pd.read_csv("dev_lookup_evaluated.csv")

# Both tails of actual price
print(df["price"].describe(percentiles=[.001,.01,.05,.5,.95,.99,.999]))
print((df["price"] < 1000).sum(), "rows with price < 1,000 AED")

# Both tails of predicted price
print((df["predicted_price"] > 1_000_000).sum(), "rows predicted > 1M AED")

# Root cause for worst offenders — check support quality, not just error
worst = df.nlargest(15, "abs_diff_pct")
print(worst[["make","model","year","trim","price","predicted_price",
             "n_neighbors","avg_similarity","provenance"]])

count    4.055300e+04
mean     1.432830e+05
std      8.028830e+06
min      8.250000e-01
0.1%     1.000000e+03
1%       2.500000e+03
5%       6.200000e+03
50%      5.000000e+04
95%      2.961000e+05
99%      7.070414e+05
99.9%    2.320655e+06
max      1.600008e+09
Name: price, dtype: float64
40 rows with price < 1,000 AED
213 rows predicted > 1M AED
             make      model    year               trim      price  \
20126         GMC     CANYON  2016.0                SLT      1.000   
14800   CHEVROLET  SILVERADO  2019.0  1500 CREW/CAB LTZ      1.000   
29594       ISUZU        NPR  2024.0                NaN      1.000   
5515       NISSAN     PATROL  2016.0                SGL      0.825   
34576       IVECO      DAILY  2018.0                 TR      1.000   
9614      HYUNDAI       HD65  2009.0                  S      1.000   
16         TOYOTA    COROLLA  2004.0            XLI MID      0.825   
3708       NISSAN     PATROL  1997.0                NaN     20.000   
6271        LEXUS  

In [3]:
import pandas as pd
dev = pd.read_csv("dev_lookup.csv")  # the source file, not the evaluated output

defenders = dev[(dev["make"] == "LAND ROVER") & (dev["model"] == "DEFENDER")]
print(defenders.sort_values("price", ascending=False).head(10)
      [["chassisNumber","year","trim","price"]])

           chassisNumber    year              trim    price
20881  SALLDVB589A772388  2009.0                90  99200.0
20908  SALEA7BU2S2376792  2025.0            LAREDO  89880.0
20782  SALEA8BU4P2177028  2023.0                SE  87020.0
21105  SALLDBMR8GA478284  2016.0               110  81800.0
20503  SALLDVB586A714615  2006.0               110  81415.0
21029  SALEA7BEXT2489075  2026.0  SE PLATINUM CITY  78300.0
20562  SALLDHSP8FA467855  2015.0               110  73800.0
21018  SALE2EEU4P2126104  2023.0                EX  73720.0
20986  SALLDHMV8TA983219  1996.0             FLEET   7300.0
21023  SALEP7EU9P2120442  2023.0            SAHARA  69200.0


In [4]:
import re

dev["price"] = (
    dev["price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(r"[^\d.\-]", "", regex=True)   # strip currency symbols, stray chars
    .replace("", pd.NA)
    .astype(float)
)

print(dev["price"].isna().sum(), "rows failed to parse")  # check before trusting the sort

print(dev.nlargest(30, "price")[["make","model","year","trim","price"]].to_string())
print(dev.nsmallest(50, "price")[["make","model","year","trim","price"]].to_string())

0 rows failed to parse
                make        model    year            trim         price
28198            ROX          NaN  2026.0             STD  1.600008e+09
20961     LAND ROVER     DEFENDER  2025.0   X DYNAMIC HSE  2.219200e+08
34627        BUGATTI       CHIRON  2022.0     SUPER SPORT  2.077600e+07
33459        FERRARI  DAYTONA SP3  2025.0             STD  1.982030e+07
34843         PAGANI       HUAYRA  2020.0     ROADSTER BC  1.869840e+07
33448        FERRARI    DAYTONA S  2025.0             STD  1.783830e+07
33377        FERRARI    LAFERRARI  2015.0             STD  1.677662e+07
34631        BUGATTI       CHIRON  2023.0             STD  1.668180e+07
34633        BUGATTI       CHIRON  2023.0             STD  1.668180e+07
33491        FERRARI    LAFERRARI  2017.0             STD  1.611690e+07
34635        BUGATTI       CHIRON  2022.0             STD  1.573750e+07
39650        BUGATTI       CHIRON  2024.0           SPORT  1.516076e+07
34630        BUGATTI       CHIRON  2024.0

In [5]:
group_cols_priority = [["make","model","year"], ["make","model"], ["make"]]
MIN_GROUP_SIZE = 5
LOWER_RATIO, UPPER_RATIO = 0.1, 10  # tune after step 2's inspection — don't take these as final

dev["price_flag"] = False
remaining = dev.copy()
for cols in group_cols_priority:
    sizes = remaining.groupby(cols)["price"].transform("size")
    med = remaining.groupby(cols)["price"].transform("median")
    eligible = sizes >= MIN_GROUP_SIZE
    bad = eligible & ((remaining["price"] < LOWER_RATIO * med) | (remaining["price"] > UPPER_RATIO * med))
    dev.loc[remaining[bad].index, "price_flag"] = True
    remaining = remaining[~eligible | ~bad]  # rows resolved at this tier drop out of the next tier's pass

In [6]:
target = dev[dev["price"] == dev["price"].max()]
print(target[["chassisNumber","make","model","year","trim","price"]].to_string())
print("make repr :", repr(target["make"].iloc[0]))
print("model repr:", repr(target["model"].iloc[0]))

           chassisNumber make model    year trim         price
28198  HJ4ABBHK5TN088892  ROX   NaN  2026.0  STD  1.600008e+09
make repr : 'ROX'
model repr: nan


In [7]:
mask = dev["make"].str.strip().str.upper() == "LAND ROVER"
print(dev.loc[mask, "model"].str.strip().str.upper().value_counts())

model
RANGE ROVER                 263
RANGE ROVER SPORT           212
DEFENDER 110                172
RANGE ROVER EVOQUE          119
DEFENDER                    104
RANGE ROVER VELAR            98
DEFENDER 90                  79
DISCOVERY SPORT              49
LR4                          35
DISCOVERY                    32
LR2                          21
DEFENDER 130                 19
LR3                          15
110                           6
DEVENDR                       3
VOGUE                         3
RANGE ROVER VOGUE             2
DEFENDER110                   2
HSE                           2
FREELANDER                    1
EC01                          1
RANGE ROVER HSE               1
ENVISION                      1
LR 3                          1
DEFENDER 90 X-DYNAMIC SE      1
RANGE ROVER VOG               1
Name: count, dtype: int64


In [8]:
def flag_price_outliers(df, group_tiers, min_group_size=5,
                         lower_ratio=0.1, upper_ratio=10.0, hard_floor=None):
    df = df.copy()
    df["price_flag"] = False
    df["flag_reason"] = ""
    remaining = df.copy()

    for cols in group_tiers:
        sizes = remaining.groupby(cols)["price"].transform("size")
        med   = remaining.groupby(cols)["price"].transform("median")
        eligible = sizes >= min_group_size
        bad = eligible & ((remaining["price"] < lower_ratio * med) |
                           (remaining["price"] > upper_ratio * med))
        df.loc[remaining[bad].index, "price_flag"] = True
        df.loc[remaining[bad].index, "flag_reason"] = f"ratio_outlier_{'_'.join(cols)}"
        remaining = remaining[~(eligible & bad) & ~eligible.where(eligible, True).eq(False)]
        remaining = remaining[eligible == False].pipe(lambda d: pd.concat([d, remaining[eligible & ~bad]]))
        # rows resolved (flagged OR cleared) at this tier exit; only "no group at this tier" rows fall through

    if hard_floor is not None:
        floor_hit = df["price"] < hard_floor
        df.loc[floor_hit, "price_flag"] = True
        df.loc[floor_hit, "flag_reason"] = df.loc[floor_hit, "flag_reason"].replace("", "below_hard_floor")

    return df

group_tiers = [["make","model","year"], ["make","model"], ["make"]]
cleaned = flag_price_outliers(dev, group_tiers, min_group_size=5,
                               lower_ratio=0.1, upper_ratio=10.0, hard_floor=500)

print(cleaned["price_flag"].sum(), "rows flagged")
print(cleaned[cleaned["price_flag"]].sort_values("price", ascending=False)
      [["make","model","year","trim","price","flag_reason"]].head(30).to_string())

1627 rows flagged
                make         model    year                     trim         price                    flag_reason
28198            ROX           NaN  2026.0                      STD  1.600008e+09             ratio_outlier_make
20961     LAND ROVER      DEFENDER  2025.0            X DYNAMIC HSE  2.219200e+08  ratio_outlier_make_model_year
33459        FERRARI   DAYTONA SP3  2025.0                      STD  1.982030e+07             ratio_outlier_make
33448        FERRARI     DAYTONA S  2025.0                      STD  1.783830e+07             ratio_outlier_make
33377        FERRARI     LAFERRARI  2015.0                      STD  1.677662e+07             ratio_outlier_make
33491        FERRARI     LAFERRARI  2017.0                      STD  1.611690e+07             ratio_outlier_make
34330        MCLAREN    SPEEDTRAIL  2020.0                      STD  1.184230e+07             ratio_outlier_make
33209    LAMBORGHINI          SIAN  2021.0                      STD  1.121900e

C:\Users\Admin\AppData\Local\Temp\ipykernel_9396\2331220713.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  remaining = remaining[eligible == False].pipe(lambda d: pd.concat([d, remaining[eligible & ~bad]]))
C:\Users\Admin\AppData\Local\Temp\ipykernel_9396\2331220713.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  remaining = remaining[eligible == False].pipe(lambda d: pd.concat([d, remaining[eligible & ~bad]]))
C:\Users\Admin\AppData\Local\Temp\ipykernel_9396\2331220713.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  remaining = remaining[eligible == False].pipe(lambda d: pd.concat([d, remaining[eligible & ~bad]]))
C:\Users\Admin\AppData\Local\Temp\ipykernel_9396\2331220713.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  remaining = remaining[eligible == False].pipe(lambda d: pd.concat([d, remaining[eligible & ~bad]]))
C:\Users\Admin\AppDa

In [9]:
def flag_price_outliers(df, group_tiers, min_group_size=5,
                         lower_ratio=0.1, upper_ratio=10.0, hard_floor=500):
    df = df.copy()
    df["price_flag"] = False
    df["flag_reason"] = ""
    resolved = pd.Index([])  # indices settled at a finer tier — don't revisit

    for cols in group_tiers:
        pool = df.drop(index=resolved)
        sizes = pool.groupby(cols)["price"].transform("size")
        med   = pool.groupby(cols)["price"].transform("median")
        eligible = sizes >= min_group_size

        too_low = eligible & (pool["price"] < lower_ratio * med)
        df.loc[pool[too_low].index, "price_flag"] = True
        df.loc[pool[too_low].index, "flag_reason"] = f"low_outlier_{'_'.join(cols)}"

        # upper bound: only ever applied at (make, model) or finer — see Fix 1
        if cols != ["make"]:
            too_high = eligible & (pool["price"] > upper_ratio * med)
            df.loc[pool[too_high].index, "price_flag"] = True
            df.loc[pool[too_high].index, "flag_reason"] = f"high_outlier_{'_'.join(cols)}"
            resolved = resolved.union(pool[eligible & (too_low | too_high)].index)
            resolved = resolved.union(pool[eligible & ~too_low & ~too_high].index)  # clean at this tier, don't re-test downstream
        else:
            resolved = resolved.union(pool[eligible & too_low].index)
            resolved = resolved.union(pool[eligible & ~too_low].index)

    # absolute floor catches placeholder garbage regardless of group, including null-model rows like ROX
    floor_hit = df["price"] < hard_floor
    df.loc[floor_hit, "price_flag"] = True
    df.loc[floor_hit & (df["flag_reason"] == ""), "flag_reason"] = "below_hard_floor"

    # rows still unresolved after all tiers (model thin even at make level, e.g. n<5 even for `make`)
    untested = df.index.difference(resolved)
    df.loc[untested, "flag_reason"] = df.loc[untested, "flag_reason"].replace("", "insufficient_data_no_flag")

    return df

In [10]:
checks = ["LAFERRARI", "P1", "SIAN", "SLR MCLAREN", "DEFENDER"]
print(cleaned[cleaned["model"].isin(checks)][["make","model","year","price","price_flag","flag_reason"]]
      .to_string())

# ROX specifically — confirm it needs the absolute ceiling, not the ratio logic
print(cleaned[cleaned["make"]=="ROX"][["model","price","price_flag","flag_reason"]])

# the bucket the ratio logic deliberately can't judge — eyeball it manually, it's short
print(cleaned[cleaned["flag_reason"]=="insufficient_data_no_flag"]
      .sort_values("price", ascending=False).head(30)
      [["make","model","year","price"]].to_string())

                make        model    year        price  price_flag                    flag_reason
11877  MERCEDES BENZ  SLR MCLAREN  2009.0    2433900.0        True             ratio_outlier_make
11887  MERCEDES BENZ  SLR MCLAREN  2009.0    9800000.0        True             ratio_outlier_make
12153  MERCEDES BENZ  SLR MCLAREN  2005.0    1008800.0        True             ratio_outlier_make
12226  MERCEDES BENZ  SLR MCLAREN  2005.0    1008800.0        True             ratio_outlier_make
12408  MERCEDES BENZ  SLR MCLAREN  2006.0     862800.0       False                               
12561  MERCEDES BENZ  SLR MCLAREN  2006.0    1063100.0        True             ratio_outlier_make
13825  MERCEDES BENZ  SLR MCLAREN  2007.0    1432800.0        True             ratio_outlier_make
14062  MERCEDES BENZ  SLR MCLAREN  2008.0    1132400.0        True             ratio_outlier_make
15672      CHEVROLET     DEFENDER  2022.0      22100.0       False                               
20338     LAND ROVER

In [11]:
def flag_price_outliers(df, group_tiers, min_group_size=5,
                         lower_ratio=0.1, upper_ratio=10.0,
                         hard_floor=500, hard_ceiling=50_000_000):
    df = df.copy()
    df["price_flag"] = False
    df["flag_reason"] = ""
    resolved = set()

    for cols in group_tiers:
        pool = df.loc[~df.index.isin(resolved)]
        sizes = pool.groupby(cols)["price"].transform("size")
        med   = pool.groupby(cols)["price"].transform("median")
        eligible = sizes >= min_group_size

        too_low = eligible & (pool["price"] < lower_ratio * med)
        is_make_only = (cols == ["make"])
        too_high = (eligible & (pool["price"] > upper_ratio * med)) if not is_make_only \
                   else pd.Series(False, index=pool.index)

        df.loc[pool[too_low].index, "price_flag"] = True
        df.loc[pool[too_low].index, "flag_reason"] = f"low_outlier_{'_'.join(cols)}"
        df.loc[pool[too_high].index, "price_flag"] = True
        df.loc[pool[too_high].index, "flag_reason"] = f"high_outlier_{'_'.join(cols)}"

        resolved.update(pool[eligible].index)  # tested at this tier — don't revisit, flagged or not

    floor_hit, ceiling_hit = df["price"] < hard_floor, df["price"] > hard_ceiling
    df.loc[floor_hit, "price_flag"] = True
    df.loc[floor_hit & (df["flag_reason"]==""), "flag_reason"] = "below_hard_floor"
    df.loc[ceiling_hit, "price_flag"] = True
    df.loc[ceiling_hit & (df["flag_reason"]==""), "flag_reason"] = "above_hard_ceiling"

    untested = ~df.index.isin(resolved) & (df["flag_reason"] == "")
    df.loc[untested, "flag_reason"] = "insufficient_data_no_flag"
    return df

cleaned = flag_price_outliers(dev, [["make","model","year"], ["make","model"], ["make"]])

In [12]:
checks = ["LAFERRARI", "P1", "SIAN", "SLR MCLAREN", "DEFENDER"]
print(cleaned[cleaned["model"].isin(checks)][["make","model","year","price","price_flag","flag_reason"]].to_string())
print(cleaned[cleaned["make"]=="ROX"][["model","price","price_flag","flag_reason"]])
print(cleaned[cleaned["flag_reason"]=="insufficient_data_no_flag"].sort_values("price", ascending=False).head(30)
      [["make","model","year","price"]].to_string())

                make        model    year        price  price_flag                   flag_reason
11877  MERCEDES BENZ  SLR MCLAREN  2009.0    2433900.0       False                              
11887  MERCEDES BENZ  SLR MCLAREN  2009.0    9800000.0       False                              
12153  MERCEDES BENZ  SLR MCLAREN  2005.0    1008800.0       False                              
12226  MERCEDES BENZ  SLR MCLAREN  2005.0    1008800.0       False                              
12408  MERCEDES BENZ  SLR MCLAREN  2006.0     862800.0       False                              
12561  MERCEDES BENZ  SLR MCLAREN  2006.0    1063100.0       False                              
13825  MERCEDES BENZ  SLR MCLAREN  2007.0    1432800.0       False                              
14062  MERCEDES BENZ  SLR MCLAREN  2008.0    1132400.0       False                              
15672      CHEVROLET     DEFENDER  2022.0      22100.0       False                              
20338     LAND ROVER     DEFEN

In [13]:
print(cleaned[(cleaned["make"]=="LAND ROVER") & (cleaned["model"]=="DEFENDER")]
      .query("year <= 2000")[["chassisNumber","year","trim","price","price_flag"]]
      .sort_values("year"))

           chassisNumber    year                  trim     price  price_flag
20868  SALLDHAD8HA481374  1991.0  1500 REG/CAB SLE Z71   19095.0       False
20869  SALLDHAD8HA481374  1991.0  1500 REG/CAB SLE Z71   45505.0       False
20775  SALLDVAY8TA988024  1996.0                   SR5   30115.0       False
20986  SALLDHMV8TA983219  1996.0                 FLEET    7300.0       False
21132  SALLDHAD8TA996532  1996.0                   STD  138900.0       False
36073                NaN  1996.0                   110   20000.0       False
21362  SALDV3242VA131588  2000.0       SUPER SAFARI T1   53200.0       False


In [14]:
clean_pool = cleaned[~cleaned["price_flag"]].copy()
print(f"{len(clean_pool):,} / {len(cleaned):,} rows retained "
      f"({cleaned['price_flag'].sum():,} excluded)")

clean_pool.to_csv("dev_lookup_filtered.csv", index=False)

40,102 / 40,553 rows retained (451 excluded)


In [15]:
manual_review_ids = [20868, 20869, 20775, 20986, 21362]  # mismatched-trim rows found this turn
cleaned.loc[cleaned.index.isin(manual_review_ids), "flag_reason"] = "trim_mismatch_manual_review"
cleaned.loc[cleaned.index.isin(manual_review_ids), "price_flag"] = True

In [16]:
# Cell 14 (fixed) — keep dupes as a separate, permanent snapshot; don't mutate `cleaned` yet
dupes = cleaned[cleaned.duplicated("chassisNumber", keep=False) & cleaned["chassisNumber"].notna()]
print(len(dupes), "rows share a chassisNumber with another row")
# NOTE: do NOT reassign `cleaned` here. Inspect first (Cell 15), decide the rule, THEN apply it.

1997 rows share a chassisNumber with another row


In [17]:
# Cell 15 (fixed) — now safe regardless of how many times you re-run it
sample_keys = dupes["chassisNumber"].value_counts().index[:10]
for key in sample_keys:
    group = dupes[dupes["chassisNumber"] == key]
    print(group[["chassisNumber", "make", "model", "year", "trim", "price"]])
    print(f"--- ({len(group)} rows) ---")

print(dupes["chassisNumber"].value_counts().describe())
print(dupes["chassisNumber"].value_counts().head(15))

           chassisNumber           make model    year  \
13281  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13282  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13283  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13284  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13285  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   

                            trim     price  
13281  FALCON EDITION EXTRA LONG  264400.0  
13282  FALCON EDITION EXTRA LONG  228600.0  
13283  FALCON EDITION EXTRA LONG  264400.0  
13284  FALCON EDITION EXTRA LONG  264400.0  
13285  FALCON EDITION EXTRA LONG  176200.0  
--- (5 rows) ---
          chassisNumber    make  model    year trim     price
1067  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX   69445.0
1068  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX   64260.0
1069  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX   38430.0
1070  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX  281960.0
--- (4 rows) ---
          chassisNumber        make   model

In [18]:
# Cell 14 (fixed) — diagnose BEFORE touching `cleaned`
dupes = cleaned[cleaned.duplicated("chassisNumber", keep=False) & cleaned["chassisNumber"].notna()]
print(len(dupes), "rows share a chassisNumber with another row")

# Cell 15 (fixed) — inspect the REAL duplicate groups, from `dupes`, not `cleaned`
sample_keys = dupes["chassisNumber"].value_counts().index[:10]
for key in sample_keys:
    group = dupes[dupes["chassisNumber"] == key]
    print(group[["chassisNumber", "make", "model", "year", "trim", "price"]])
    print(f"--- ({len(group)} rows) ---")

# Check: is this driven by a handful of placeholder values repeated many times,
# or many distinct chassis values each duplicated exactly once/twice?
print(dupes["chassisNumber"].value_counts().describe())
print(dupes["chassisNumber"].value_counts().head(15))  # which values repeat MOST

1997 rows share a chassisNumber with another row
           chassisNumber           make model    year  \
13281  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13282  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13283  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13284  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   
13285  W1WV0LEZ3P4118307  MERCEDES BENZ  V250  2023.0   

                            trim     price  
13281  FALCON EDITION EXTRA LONG  264400.0  
13282  FALCON EDITION EXTRA LONG  228600.0  
13283  FALCON EDITION EXTRA LONG  264400.0  
13284  FALCON EDITION EXTRA LONG  264400.0  
13285  FALCON EDITION EXTRA LONG  176200.0  
--- (5 rows) ---
          chassisNumber    make  model    year trim     price
1067  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX   69445.0
1068  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX   64260.0
1069  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX   38430.0
1070  WBWB0111111111111  TOYOTA  CAMRY  2009.0  GLX  281960.0
--- (4 rows

In [19]:
import re

def is_placeholder(chassis):
    if pd.isna(chassis):
        return False
    s = str(chassis)
    # 6+ consecutive identical characters anywhere = strong placeholder signal
    if re.search(r'(.)\1{5,}', s):
        return True
    # extremely low character diversity overall (e.g. only 2-3 distinct chars in a 17-char string)
    if len(set(s)) <= 3 and len(s) >= 10:
        return True
    return False

dupes["is_placeholder_chassis"] = dupes["chassisNumber"].apply(is_placeholder)
print(dupes["is_placeholder_chassis"].sum(), "duplicate rows involve placeholder chassis values")
print(dupes[dupes["is_placeholder_chassis"]]["chassisNumber"].unique())

10 duplicate rows involve placeholder chassis values
<ArrowStringArray>
['WBWB0111111111111', 'W1111111111111111', '0000NNNNN11111111',
 'ZFA18300000043173']
Length: 4, dtype: str


In [20]:
genuine_dupes = dupes[~dupes["is_placeholder_chassis"]]
median_price_by_chassis = genuine_dupes.groupby("chassisNumber")["price"].median()

# replace price with the group median, then drop to 1 row per chassis
cleaned.loc[cleaned["chassisNumber"].isin(median_price_by_chassis.index), "price"] = \
    cleaned["chassisNumber"].map(median_price_by_chassis)
cleaned = cleaned.drop_duplicates("chassisNumber", keep="first")

In [21]:
print(is_placeholder("WBWB0111111111111"))  # should be True
print(is_placeholder("5TDKK3DC4ES425556"))  # should be False — real Toyota VIN
print(is_placeholder("ZAMKM45B000054349"))  # should be False — real Maserati VIN, has some repetition but not 6+ run

True
False
False


In [22]:
print(dupes[dupes["chassisNumber"] == "ZFA18300000043173"]
      [["chassisNumber","make","model","year","trim","price"]])

           chassisNumber  make      model    year trim    price
33264  ZFA18300000043173  FIAT  BARCHETTA  2000.0  STD  41100.0
33265  ZFA18300000043173  FIAT  BARCHETTA  2000.0  STD  38300.0


In [23]:
def is_placeholder(chassis):
    if pd.isna(chassis):
        return False
    s = str(chassis)
    if re.search(r'(.)\1{5,}', s):
        return True
    if len(set(s)) <= 3 and len(s) >= 10:
        return True
    return False

In [24]:
dupes["is_placeholder_chassis"] = dupes["chassisNumber"].apply(is_placeholder)

# Manual override — confirmed false positive (real Fiat VIN, plausible re-listing)
dupes.loc[dupes["chassisNumber"] == "ZFA18300000043173", "is_placeholder_chassis"] = False

print(dupes["is_placeholder_chassis"].sum(), "rows confirmed placeholder")
print(dupes[dupes["is_placeholder_chassis"]]["chassisNumber"].unique())

8 rows confirmed placeholder
<ArrowStringArray>
['WBWB0111111111111', 'W1111111111111111', '0000NNNNN11111111']
Length: 3, dtype: str


In [25]:
genuine_dupes = dupes[~dupes["is_placeholder_chassis"]]
median_price_by_chassis = genuine_dupes.groupby("chassisNumber")["price"].median()

mask = cleaned["chassisNumber"].isin(median_price_by_chassis.index)
cleaned.loc[mask, "price"] = cleaned.loc[mask, "chassisNumber"].map(median_price_by_chassis)
cleaned = cleaned.drop_duplicates("chassisNumber", keep="first")  # safe now — price already set to median

# Placeholder rows: do NOT dedup (they're likely distinct vehicles), but flag chassisNumber as unreliable
placeholder_values = dupes[dupes["is_placeholder_chassis"]]["chassisNumber"].unique()
cleaned["chassis_unreliable"] = cleaned["chassisNumber"].isin(placeholder_values)

print(len(cleaned), "rows after dedup")
print(cleaned["chassis_unreliable"].sum(), "rows flagged with unreliable chassisNumber")

33940 rows after dedup
3 rows flagged with unreliable chassisNumber


In [26]:
# 1. Rows with missing chassisNumber — never touch, no dedup possible/meaningful
no_chassis = cleaned[cleaned["chassisNumber"].isna()]

# 2. Rows with a chassisNumber — split into placeholder vs. real
has_chassis = cleaned[cleaned["chassisNumber"].notna()]
is_placeholder_mask = has_chassis["chassisNumber"].isin(placeholder_values)

placeholder_rows = has_chassis[is_placeholder_mask]          # keep ALL, untouched
real_rows        = has_chassis[~is_placeholder_mask].copy()  # dedup candidates

# 3. Apply median-price collapse ONLY to real, genuinely duplicated chassisNumbers
mask = real_rows["chassisNumber"].isin(median_price_by_chassis.index)
real_rows.loc[mask, "price"] = real_rows.loc[mask, "chassisNumber"].map(median_price_by_chassis)
real_rows = real_rows.drop_duplicates("chassisNumber", keep="first")

# 4. Recombine
cleaned = pd.concat([no_chassis, placeholder_rows, real_rows], ignore_index=False)
cleaned["chassis_unreliable"] = cleaned["chassisNumber"].isin(placeholder_values)

print(len(cleaned), "rows after corrected dedup")
print(len(no_chassis), "rows had no chassisNumber (untouched)")
print(len(placeholder_rows), "placeholder rows retained untouched")
print(cleaned["chassis_unreliable"].sum(), "rows flagged unreliable (should be 8, not 3)")

33940 rows after corrected dedup
1 rows had no chassisNumber (untouched)
3 placeholder rows retained untouched
3 rows flagged unreliable (should be 8, not 3)


In [27]:
import pandas as pd
dev = pd.read_csv("dev_lookup.csv")
# ... re-run your price-cleaning steps (comma-strip, outlier flagging, manual trim-mismatch flags) ...
# up through Cell 13 — do NOT run the drop_duplicates/concat dedup logic yet

print(len(cleaned), "total rows BEFORE any dedup")
print(cleaned["chassisNumber"].isna().sum(), "rows with null chassisNumber")

dupes = cleaned[cleaned.duplicated("chassisNumber", keep=False) & cleaned["chassisNumber"].notna()]
print(len(dupes), "rows share a chassisNumber with another row")

33940 total rows BEFORE any dedup
1 rows with null chassisNumber
0 rows share a chassisNumber with another row


In [28]:
import pandas as pd
dev = pd.read_csv("dev_lookup.csv")
print(len(dev), "rows in raw dev_lookup.csv")  # must be 40,553

40553 rows in raw dev_lookup.csv


In [48]:
import os, time
print("PID:", os.getpid())

import pandas as pd
dev = pd.read_csv("dev_lookup.csv")
print(len(dev), "rows in raw dev_lookup.csv")

# --- re-run your price cleaning here (comma/currency strip from Cell 2) ---
import re
dev["price"] = (
    dev["price"].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(r"[^\d.\-]", "", regex=True)
    .replace("", pd.NA)
    .astype(float)
)

# --- re-run your outlier flagging (Cell 9's flag_price_outliers + Cell 13's manual flags) ---
# [paste your actual flag_price_outliers function + call + manual_review_ids block here]

cleaned = dev.copy()  # or whatever your flagging step assigns to `cleaned`

print(len(cleaned), "total rows BEFORE any dedup")
print(cleaned["chassisNumber"].isna().sum(), "rows with null chassisNumber")

dupes = cleaned[cleaned.duplicated("chassisNumber", keep=False) & cleaned["chassisNumber"].notna()]
print(len(dupes), "rows share a chassisNumber with another row")

PID: 9396
40553 rows in raw dev_lookup.csv
40553 total rows BEFORE any dedup
5572 rows with null chassisNumber
1997 rows share a chassisNumber with another row


In [49]:
import re

def is_placeholder(chassis):
    if pd.isna(chassis):
        return False
    s = str(chassis)
    if re.search(r'(.)\1{5,}', s):
        return True
    if len(set(s)) <= 3 and len(s) >= 10:
        return True
    return False

dupes["is_placeholder_chassis"] = dupes["chassisNumber"].apply(is_placeholder)
dupes.loc[dupes["chassisNumber"] == "ZFA18300000043173", "is_placeholder_chassis"] = False  # confirmed real VIN

placeholder_values = dupes[dupes["is_placeholder_chassis"]]["chassisNumber"].unique()
print(len(placeholder_values), "placeholder chassis values:", list(placeholder_values))

genuine_dupes = dupes[~dupes["is_placeholder_chassis"]]
median_price_by_chassis = genuine_dupes.groupby("chassisNumber")["price"].median()

no_chassis = cleaned[cleaned["chassisNumber"].isna()]
has_chassis = cleaned[cleaned["chassisNumber"].notna()]
is_placeholder_mask = has_chassis["chassisNumber"].isin(placeholder_values)

placeholder_rows = has_chassis[is_placeholder_mask]
real_rows = has_chassis[~is_placeholder_mask].copy()

mask = real_rows["chassisNumber"].isin(median_price_by_chassis.index)
real_rows.loc[mask, "price"] = real_rows.loc[mask, "chassisNumber"].map(median_price_by_chassis)
real_rows = real_rows.drop_duplicates("chassisNumber", keep="first")

cleaned_deduped = pd.concat([no_chassis, placeholder_rows, real_rows], ignore_index=False)
cleaned_deduped["chassis_unreliable"] = cleaned_deduped["chassisNumber"].isin(placeholder_values)

print(len(cleaned_deduped), "rows after corrected dedup (expect ~39,511)")
print(len(no_chassis), "null-chassis rows retained untouched (expect 5,572)")
print(len(placeholder_rows), "placeholder rows retained untouched (expect 8)")
print(cleaned_deduped["chassis_unreliable"].sum(), "rows flagged unreliable (expect 8)")

3 placeholder chassis values: ['WBWB0111111111111', 'W1111111111111111', '0000NNNNN11111111']
39516 rows after corrected dedup (expect ~39,511)
5572 null-chassis rows retained untouched (expect 5,572)
8 placeholder rows retained untouched (expect 8)
8 rows flagged unreliable (expect 8)


In [32]:
print(cleaned.columns.tolist())


['chassisNumber', 'year', 'make', 'model', 'trim', 'price']


In [50]:
# 1. Outlier flagging
def flag_price_outliers(df, group_tiers, min_group_size=5,
                         lower_ratio=0.1, upper_ratio=10.0,
                         hard_floor=500, hard_ceiling=50_000_000):
    df = df.copy()
    df["price_flag"] = False
    df["flag_reason"] = ""
    resolved = set()

    for cols in group_tiers:
        pool = df.loc[~df.index.isin(resolved)]
        sizes = pool.groupby(cols)["price"].transform("size")
        med   = pool.groupby(cols)["price"].transform("median")
        eligible = sizes >= min_group_size

        too_low = eligible & (pool["price"] < lower_ratio * med)
        is_make_only = (cols == ["make"])
        too_high = (eligible & (pool["price"] > upper_ratio * med)) if not is_make_only \
                   else pd.Series(False, index=pool.index)

        df.loc[pool[too_low].index, "price_flag"] = True
        df.loc[pool[too_low].index, "flag_reason"] = f"low_outlier_{'_'.join(cols)}"
        df.loc[pool[too_high].index, "price_flag"] = True
        df.loc[pool[too_high].index, "flag_reason"] = f"high_outlier_{'_'.join(cols)}"

        resolved.update(pool[eligible].index)

    floor_hit, ceiling_hit = df["price"] < hard_floor, df["price"] > hard_ceiling
    df.loc[floor_hit, "price_flag"] = True
    df.loc[floor_hit & (df["flag_reason"]==""), "flag_reason"] = "below_hard_floor"
    df.loc[ceiling_hit, "price_flag"] = True
    df.loc[ceiling_hit & (df["flag_reason"]==""), "flag_reason"] = "above_hard_ceiling"

    untested = ~df.index.isin(resolved) & (df["flag_reason"] == "")
    df.loc[untested, "flag_reason"] = "insufficient_data_no_flag"
    return df

cleaned = flag_price_outliers(dev, [["make","model","year"], ["make","model"], ["make"]])

manual_review_ids = [20868, 20869, 20775, 20986, 21362]
cleaned.loc[cleaned.index.isin(manual_review_ids), "flag_reason"] = "trim_mismatch_manual_review"
cleaned.loc[cleaned.index.isin(manual_review_ids), "price_flag"] = True

print(cleaned["price_flag"].sum(), "rows flagged after outlier + manual review")

# 2. Duplicate detection (on the freshly-flagged `cleaned`)
dupes = cleaned[cleaned.duplicated("chassisNumber", keep=False) & cleaned["chassisNumber"].notna()]
print(len(dupes), "rows share a chassisNumber with another row (expect 1997)")

# 3. Placeholder detection
import re
def is_placeholder(chassis):
    if pd.isna(chassis):
        return False
    s = str(chassis)
    if re.search(r'(.)\1{5,}', s):
        return True
    if len(set(s)) <= 3 and len(s) >= 10:
        return True
    return False

dupes = dupes.copy()
dupes["is_placeholder_chassis"] = dupes["chassisNumber"].apply(is_placeholder)
dupes.loc[dupes["chassisNumber"] == "ZFA18300000043173", "is_placeholder_chassis"] = False

placeholder_values = dupes[dupes["is_placeholder_chassis"]]["chassisNumber"].unique()
print(len(placeholder_values), "placeholder values (expect 3)")

# 4. Three-bucket dedup
genuine_dupes = dupes[~dupes["is_placeholder_chassis"]]
median_price_by_chassis = genuine_dupes.groupby("chassisNumber")["price"].median()

no_chassis = cleaned[cleaned["chassisNumber"].isna()]
has_chassis = cleaned[cleaned["chassisNumber"].notna()]
is_placeholder_mask = has_chassis["chassisNumber"].isin(placeholder_values)

placeholder_rows = has_chassis[is_placeholder_mask]
real_rows = has_chassis[~is_placeholder_mask].copy()

mask = real_rows["chassisNumber"].isin(median_price_by_chassis.index)
real_rows.loc[mask, "price"] = real_rows.loc[mask, "chassisNumber"].map(median_price_by_chassis)
real_rows = real_rows.drop_duplicates("chassisNumber", keep="first")

cleaned_deduped = pd.concat([no_chassis, placeholder_rows, real_rows], ignore_index=False)
cleaned_deduped["chassis_unreliable"] = cleaned_deduped["chassisNumber"].isin(placeholder_values)

print(len(cleaned_deduped), "rows after dedup (expect 39516)")

# 5. Save final filtered file
clean_pool = cleaned_deduped[~cleaned_deduped["price_flag"]].copy()
print(f"{len(clean_pool):,} / {len(cleaned_deduped):,} rows retained "
      f"({cleaned_deduped['price_flag'].sum():,} excluded)")
clean_pool.to_csv("dev_lookup_filtered.csv", index=False)

456 rows flagged after outlier + manual review
1997 rows share a chassisNumber with another row (expect 1997)
3 placeholder values (expect 3)
39516 rows after dedup (expect 39516)
39,079 / 39,516 rows retained (437 excluded)


In [34]:
unresolved_filter = cleaned_deduped[cleaned_deduped["flag_reason"] == "insufficient_data_no_flag"].copy()
print(len(unresolved_filter), "rows never reached a ratio check at any tier")

print("\n--- Bottom 20 by price (cheapest, most likely junk) ---")
print(unresolved_filter.sort_values("price").head(20)
      [["make","model","year","trim","price"]].to_string())

print("\n--- Top 20 by price (sanity check the other tail too) ---")
print(unresolved_filter.sort_values("price", ascending=False).head(20)
      [["make","model","year","trim","price"]].to_string())

print("\n--- Price distribution of this bucket ---")
print(unresolved_filter["price"].describe(percentiles=[.01,.05,.1,.5,.9,.99]))

158 rows never reached a ratio check at any tier

--- Bottom 20 by price (cheapest, most likely junk) ---
          make     model    year      trim    price
34875  PONTIAC  TRANS AM  1990.0         G    900.0
34804  MAEXTRO      S800  2025.0         S    950.0
34871   BAOJUN       NaN  2024.0       STD   1000.0
34808      VGV  U75 PLUS  2023.0       NaN   2000.0
34407     TATA   LPO1618  2026.0         S   2150.0
34411     TATA     LP613  2008.0       STD   5000.0
34786  PONTIAC  FIREBIRD  1999.0       STD   5565.0
34823   PROTON     GEN-2  2011.0  BASELINE   5700.0
34846   SATURN       SKY  2007.0  RED LINE   8300.0
34848   SATURN       SKY  2007.0  RED LINE   8300.0
34877   LANCIA     DELTA  2005.0       STD   8900.0
34827   PROTON      SAGA  2012.0   MIDLINE   9200.0
34826   PROTON      SAGA  2012.0   MIDLINE   9200.0
34224      ZNA    YUMSUN  2015.0       STD   9700.0
34824   PROTON   PERSONA  2014.0   MIDLINE  11500.0
34498     SEAT      EXEO  2011.0       STD  12200.0
34796   LU

In [35]:
evaluated = pd.read_csv("dev_lookup_filtered_evaluated.csv")

# tag which rows came from the thin-data bucket
thin_index = unresolved_filter.index
evaluated["is_thin_data"] = evaluated.index.isin(thin_index)

resolved = evaluated.dropna(subset=["predicted_price"])  # adjust if your "resolved" flag is a column instead

for label, subset in [("All resolved", resolved),
                       ("Excluding thin-data rows", resolved[~resolved["is_thin_data"]]),
                       ("Thin-data rows only", resolved[resolved["is_thin_data"]])]:
    mae = subset["abs_diff_pct"].notna()  # placeholder — use your actual abs error column name
    print(f"\n{label}: n={len(subset)}")
    print("MAE:", subset["abs_error"].mean() if "abs_error" in subset else "n/a — check column name")
    print("MAPE:", subset["abs_diff_pct"].mean())
    print("Median APE:", subset["abs_diff_pct"].median())


All resolved: n=38480
MAE: n/a — check column name
MAPE: 27.470484407484406
Median APE: 13.12

Excluding thin-data rows: n=38340
MAE: n/a — check column name
MAPE: 27.496648148148147
Median APE: 13.13

Thin-data rows only: n=140
MAE: n/a — check column name
MAPE: 20.305357142857144
Median APE: 10.225


In [36]:
# adjust the condition below to however your evaluator marks "unresolved" —
# likely a null predicted_price, or an explicit status column
unresolved_rows = evaluated[evaluated["predicted_price"].isna()]  # or evaluated["status"] == "unresolved"
print(len(unresolved_rows), "unresolved rows")
print(unresolved_rows[["make","model","year","trim","n_neighbors","avg_similarity"]].describe(include="all"))
print(unresolved_rows["make"].value_counts().head(15))

599 unresolved rows
                 make model         year trim  n_neighbors  avg_similarity
count             599   599   599.000000  563        599.0           599.0
unique            124   586          NaN  199          NaN             NaN
top     MERCEDES BENZ     7          NaN  STD          NaN             NaN
freq               68     3          NaN  196          NaN             NaN
mean              NaN   NaN  2016.185309  NaN          0.0             0.0
std               NaN   NaN    10.604349  NaN          0.0             0.0
min               NaN   NaN  1965.000000  NaN          0.0             0.0
25%               NaN   NaN  2011.000000  NaN          0.0             0.0
50%               NaN   NaN  2020.000000  NaN          0.0             0.0
75%               NaN   NaN  2025.000000  NaN          0.0             0.0
max               NaN   NaN  2027.000000  NaN          0.0             0.0
make
MERCEDES BENZ    68
BMW              31
TOYOTA           29
NISSAN         

In [37]:
unresolved_sample = unresolved_rows.sort_values("year", ascending=False)
print(unresolved_sample[["make","model","year","trim"]].head(30).to_string())

# specifically check: how many unresolved rows have year >= 2026?
print((unresolved_rows["year"] >= 2026).sum(), "unresolved rows have year >= 2026")

# and: of the Mercedes/BMW/Toyota unresolved rows specifically, what does trim/year look like?
print(unresolved_rows[unresolved_rows["make"]=="MERCEDES BENZ"][["model","year","trim"]].to_string())

                make                model    year            trim
34903          HAVAL                   V7  2027.0     HIGH DELUXE
33361          GEELY                CRUZE  2027.0              LS
38955            TAM                  C12  2027.0               S
38931          LUCID              GRAVITY  2026.0         TOURING
1524       LYNK & CO                    6  2026.0      HYPER HALO
5503   MERCEDES BENZ             EQG G580  2026.0            BASE
38674       MAHINDRA           SCORPIO S6  2026.0             STD
38656         JAECOO            JAECOO J5  2026.0             TDL
38654         JAECOO            JAECOO J7  2026.0         COMFORT
5516             BYD                   E2  2026.0          LUXURY
2674             JMC              PICK UP  2026.0             STD
5502            BAIC                 X 35  2026.0          LUXURY
5501         BESTUNE              NAT E05  2026.0  STD  (NON GCC)
38645       DONGFENG          ZN1032UCM5A  2026.0             STD
37180     

In [38]:
for m in unresolved_rows[unresolved_rows["make"]=="MERCEDES BENZ"]["model"].head(20):
    count = (cleaned_deduped[(cleaned_deduped["make"]=="MERCEDES BENZ") & (cleaned_deduped["model"]==m)]).shape[0]
    print(f"{m!r}: {count} rows in dataset with this exact model")

'CLK230 K': 2 rows in dataset with this exact model
'B150': 1 rows in dataset with this exact model
'EQG G580': 1 rows in dataset with this exact model
'V280': 1 rows in dataset with this exact model
'ML430': 1 rows in dataset with this exact model
'V220 D': 1 rows in dataset with this exact model
'G55': 1 rows in dataset with this exact model
'S400H': 1 rows in dataset with this exact model
'GLE580': 1 rows in dataset with this exact model
'CLA180': 1 rows in dataset with this exact model
'G650 MAYBACH': 1 rows in dataset with this exact model
'CLC180K': 1 rows in dataset with this exact model
'SL43': 1 rows in dataset with this exact model
'GLA43 AMG': 1 rows in dataset with this exact model
'EQS 53 AMG': 1 rows in dataset with this exact model
'A45 AMG': 1 rows in dataset with this exact model
'500 SEL': 1 rows in dataset with this exact model
'ML550': 1 rows in dataset with this exact model
'B250 E': 1 rows in dataset with this exact model
'CLE450': 1 rows in dataset with this exac

In [39]:
import pandas as pd
import numpy as np

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")

# ── Step 0: Orient ────────────────────────────────────────────────────────────
print("=== COLUMNS ===")
print(df.columns.tolist())
print(f"\nTotal rows: {len(df):,}")
print(f"dtypes sample:\n{df.dtypes}")

=== COLUMNS ===
['chassisNumber', 'year', 'make', 'model', 'trim', 'price', 'price_flag', 'flag_reason', 'chassis_unreliable', 'predicted_price', 'price_diff', 'price_diff_pct', 'abs_diff', 'abs_diff_pct', 'n_neighbors', 'avg_similarity', 'provenance']

Total rows: 39,079
dtypes sample:
chassisNumber             str
year                  float64
make                      str
model                     str
trim                      str
price                 float64
price_flag               bool
flag_reason               str
chassis_unreliable       bool
predicted_price       float64
price_diff            float64
price_diff_pct        float64
abs_diff              float64
abs_diff_pct          float64
n_neighbors             int64
avg_similarity        float64
provenance                str
dtype: object


In [40]:
import pandas as pd
import numpy as np

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")

resolved = df[df["predicted_price"].notna()].copy()
unresolved = df[df["predicted_price"].isna()].copy()

print(f"Resolved:   {len(resolved):,}  ({100*len(resolved)/len(df):.1f}%)")
print(f"Unresolved: {len(unresolved):,}  ({100*len(unresolved)/len(df):.1f}%)")

# ── Overall metrics ───────────────────────────────────────────────────────────
def metrics(subset, label):
    if len(subset) == 0:
        print(f"\n{label}: NO ROWS")
        return
    print(f"\n{'='*55}")
    print(f"{label}  (n={len(subset):,})")
    print(f"{'='*55}")
    print(f"  MAE         : {subset['abs_diff'].mean():>14,.0f} AED")
    print(f"  MAPE        : {subset['abs_diff_pct'].mean():>13.2f}%")
    print(f"  Median APE  : {subset['abs_diff_pct'].median():>13.2f}%")
    print(f"  p90 APE     : {subset['abs_diff_pct'].quantile(.90):>13.2f}%")
    print(f"  p95 APE     : {subset['abs_diff_pct'].quantile(.95):>13.2f}%")
    print(f"  Avg sim     : {subset['avg_similarity'].mean():>13.3f}")
    print(f"  Avg neighbors: {subset['n_neighbors'].mean():>12.1f}")

metrics(resolved, "ALL RESOLVED")

# ── By provenance tier ────────────────────────────────────────────────────────
print("\n\n=== BY PROVENANCE TIER ===")
for tier, grp in resolved.groupby("provenance"):
    metrics(grp, f"provenance={tier}")

# ── Worst offenders ───────────────────────────────────────────────────────────
print("\n\n=== TOP 20 WORST APE ROWS ===")
print(resolved.nlargest(20, "abs_diff_pct")
      [["make","model","year","trim","price","predicted_price",
        "abs_diff_pct","n_neighbors","avg_similarity","provenance"]]
      .to_string())

# ── Unresolved breakdown ──────────────────────────────────────────────────────
print(f"\n\n=== UNRESOLVED BREAKDOWN (n={len(unresolved):,}) ===")
print(unresolved["make"].value_counts().head(15).to_string())

Resolved:   38,480  (98.5%)
Unresolved: 599  (1.5%)

ALL RESOLVED  (n=38,480)
  MAE         :         17,273 AED
  MAPE        :         27.47%
  Median APE  :         13.12%
  p90 APE     :         47.85%
  p95 APE     :         77.35%
  Avg sim     :         0.907
  Avg neighbors:          9.5


=== BY PROVENANCE TIER ===

provenance=exact  (n=38,248)
  MAE         :         17,183 AED
  MAPE        :         26.85%
  Median APE  :         13.06%
  p90 APE     :         47.17%
  p95 APE     :         75.67%
  Avg sim     :         0.908
  Avg neighbors:          9.5

provenance=make_fallback  (n=232)
  MAE         :         31,978 AED
  MAPE        :        129.15%
  Median APE  :         41.41%
  p90 APE     :        208.29%
  p95 APE     :        412.88%
  Avg sim     :         0.731
  Avg neighbors:          9.8


=== TOP 20 WORST APE ROWS ===
                make     model    year            trim    price  predicted_price  abs_diff_pct  n_neighbors  avg_similarity     provenance


In [41]:
import pandas as pd
import numpy as np

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")
unresolved = df[df["predicted_price"].isna()].copy()

# ── Year distribution ─────────────────────────────────────────────────────────
print("=== YEAR DISTRIBUTION (unresolved) ===")
print(unresolved["year"].describe(percentiles=[.01,.05,.1,.5,.9,.95,.99]))
print(f"\nyear >= 2026: {(unresolved['year'] >= 2026).sum()}")
print(f"year <= 1990: {(unresolved['year'] <= 1990).sum()}")
print(f"year is null: {unresolved['year'].isna().sum()}")

# ── Model row counts — how thin are these in the full dataset? ────────────────
print("\n\n=== ROW COUNT IN FULL DATASET FOR UNRESOLVED (make, model) PAIRS ===")
full = df.copy()  # use full filtered file as the pool
pair_counts = full.groupby(["make","model"]).size().rename("pool_size")

unresolved_pairs = (
    unresolved[["make","model"]]
    .drop_duplicates()
    .merge(pair_counts.reset_index(), on=["make","model"], how="left")
    .sort_values("pool_size")
)
print(unresolved_pairs.head(30).to_string())
print(f"\nPairs with pool_size == 1: {(unresolved_pairs['pool_size'] == 1).sum()}")
print(f"Pairs with pool_size <= 3: {(unresolved_pairs['pool_size'] <= 3).sum()}")
print(f"Pairs with pool_size is null: {unresolved_pairs['pool_size'].isna().sum()}")

# ── Spot check the mainstream makes ──────────────────────────────────────────
print("\n\n=== UNRESOLVED MERCEDES / BMW / TOYOTA — year + model ===")
for make in ["MERCEDES BENZ", "BMW", "TOYOTA"]:
    sub = unresolved[unresolved["make"] == make]
    print(f"\n{make} (n={len(sub)}):")
    print(sub[["model","year","trim"]].sort_values("year").to_string())

=== YEAR DISTRIBUTION (unresolved) ===
count     599.000000
mean     2016.185309
std        10.604349
min      1965.000000
1%       1984.000000
5%       1992.000000
10%      2001.000000
50%      2020.000000
90%      2025.000000
95%      2026.000000
99%      2026.000000
max      2027.000000
Name: year, dtype: float64

year >= 2026: 59
year <= 1990: 27
year is null: 0


=== ROW COUNT IN FULL DATASET FOR UNRESOLVED (make, model) PAIRS ===
             make                 model  pool_size
0       LYNK & CO                     6          1
1             JMC               PICK UP          1
2          NISSAN        PATROL PICK UP          1
3          TOYOTA  LAND CRUISER PICK UP          1
4        DONGFENG                     7          1
5           OMODA                     7          1
6         HYUNDAI                 COUPE          1
7             BMW       420I GRAN COUPE          1
8             KIA           CERATO KOUP          1
9        FORTHING                    S7          1

In [42]:
import pandas as pd
import re

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")

# ── Pattern 1: Chassis/spec strings leaked into model field ───────────────────
# Signals: displacement strings, 4X4, DC/SC cab specs, raw chassis prefixes
chassis_pattern = re.compile(
    r'(\d+\.\d+\s*L)'           # displacement e.g. 2.7L
    r'|(\b4X4\b)'
    r'|(\bDC\b|\bSC\b)'         # double/single cab abbreviations
    r'|(\bAT\b|\bMT\b)'         # transmission codes
    r'|(\d{2}HX\b)'             # chassis prefix e.g. 26HX
    r'|(\bHX\b)',
    re.IGNORECASE
)

df["model_is_chassis_string"] = (
    df["model"].notna() &
    df["model"].str.contains(chassis_pattern, regex=True)
)

print("=== PATTERN 1: Chassis/spec string in model field ===")
print(f"Rows affected: {df['model_is_chassis_string'].sum()}")
print(df[df["model_is_chassis_string"]][["make","model","year","price"]]
      .value_counts(["make","model"]).head(30).to_string())

# ── Pattern 2: Known cross-make contamination ─────────────────────────────────
# Models confirmed to belong to a different make than listed
cross_make = {
    "TOYOTA":    ["PARISIENNE BROUGHAM", "H3", "GS300", "ES8", "IQ"],
    "BMW":       ["IQ", "CAR"],
    "HONDA":     ["ACURA"],   # from VIN lessons learned
    "FORD":      ["LINCOLN"],
    "MERCEDES BENZ": ["SMART"],
}

contaminated_idx = set()
for make, bad_models in cross_make.items():
    mask = (df["make"] == make) & (df["model"].isin(bad_models))
    contaminated_idx.update(df[mask].index.tolist())
    if mask.sum() > 0:
        print(f"\n{make} — cross-make models: {mask.sum()} rows")
        print(df[mask][["make","model","year","price"]].to_string())

# ── Pattern 3: Garbage model entries ─────────────────────────────────────────
# Pure numbers, single chars, known placeholders
garbage_pattern = re.compile(r'^\d+$|^[A-Z]$|^NaN$|^STD$|^N/A$', re.IGNORECASE)
df["model_is_garbage"] = (
    df["model"].notna() &
    df["model"].str.strip().str.match(garbage_pattern)
)

# Also catch blank/whitespace-only
df["model_is_garbage"] |= df["model"].str.strip().eq("")

print("\n\n=== PATTERN 3: Garbage model entries ===")
print(f"Rows affected: {df['model_is_garbage'].sum()}")
print(df[df["model_is_garbage"]][["make","model","year","price"]]
      .value_counts(["make","model"]).head(20).to_string())

# ── Summary ───────────────────────────────────────────────────────────────────
any_contaminated = (
    df["model_is_chassis_string"] |
    df.index.isin(contaminated_idx) |
    df["model_is_garbage"]
)
print(f"\n\n=== SUMMARY ===")
print(f"Total rows:              {len(df):,}")
print(f"Chassis string in model: {df['model_is_chassis_string'].sum():,}")
print(f"Cross-make contamination:{len(contaminated_idx):,}")
print(f"Garbage model entries:   {df['model_is_garbage'].sum():,}")
print(f"Total contaminated:      {any_contaminated.sum():,}  (may overlap)")

=== PATTERN 1: Chassis/spec string in model field ===
Rows affected: 9
make        model              
LEXUS       SC 300                 2
TOYOTA      26HX DC 4X4P2.7LAT     1
            26HX DC 4X4D2.4L AT    1
            HX DC 4X4 D 2.4L AT    1
NISSAN      MT                     1
VOLKSWAGEN  JETTA 2.5L             1
PEUGEOT     LAND REK - X MT        1
MAHINDRA    SCORPIO SC S6          1

TOYOTA — cross-make models: 15 rows
        make                model    year    price
5658  TOYOTA                   IQ  2013.0  12800.0
5863  TOYOTA                   IQ  2012.0  10700.0
5981  TOYOTA                   IQ  2013.0  11600.0
6122  TOYOTA                   IQ  2014.0  14500.0
6354  TOYOTA                   IQ  2014.0  14300.0
6541  TOYOTA                   IQ  2012.0  11500.0
6963  TOYOTA                   IQ  2014.0  32800.0
8051  TOYOTA                   IQ  2012.0  12500.0
8152  TOYOTA                   IQ  2012.0  12500.0
8232  TOYOTA                   IQ  2014.0  15500.0
838

C:\Users\Admin\AppData\Local\Temp\ipykernel_9396\1020204977.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["model"].str.contains(chassis_pattern, regex=True)


In [43]:
# Verify Toyota IQ legitimacy
print(df[(df["make"]=="TOYOTA") & (df["model"]=="IQ")]
      [["make","model","year","price"]].describe())

# Check how many rows would actually be removed under the corrected scope
confirmed_bad = {
    "TOYOTA": ["GS300", "PARISIENNE BROUGHAM", "ES8", "H3"],
    "BMW":    ["CAR", "IQ"],
}
chassis_bad_idx = df[df["model_is_chassis_string"] & 
                     ~df["model"].isin(["SC 300","SCORPIO SC S6"])].index

cross_bad_idx = set()
for make, models in confirmed_bad.items():
    cross_bad_idx.update(df[(df["make"]==make) & (df["model"].isin(models))].index)

total_bad = chassis_bad_idx.union(cross_bad_idx)
print(f"\nConfirmed contaminated rows to remove: {len(total_bad)}")
print(df.loc[total_bad][["make","model","year","price"]].to_string())

              year         price
count    11.000000     11.000000
mean   2012.909091  14354.545455
std       0.943880   6375.949127
min    2012.000000   9200.000000
25%    2012.000000  11550.000000
50%    2013.000000  12500.000000
75%    2014.000000  14400.000000
max    2014.000000  32800.000000

Confirmed contaminated rows to remove: 12
             make                model    year     price
6649       TOYOTA   26HX DC 4X4P2.7LAT  2026.0  112770.0
8381       TOYOTA                GS300  2008.0   11115.0
8421       TOYOTA  26HX DC 4X4D2.4L AT  2026.0    9300.0
8497       TOYOTA  HX DC 4X4 D 2.4L AT  2026.0   96000.0
8535       TOYOTA  PARISIENNE BROUGHAM  1986.0    9000.0
8569       TOYOTA                  ES8  2024.0   31160.0
8611       TOYOTA                   H3  2006.0   73700.0
10472      NISSAN                   MT  2007.0    7100.0
22789         BMW                  CAR  2020.0  150000.0
22896         BMW                   IQ  2012.0   52500.0
26444  VOLKSWAGEN           JETTA

In [ ]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")

CONTAMINATED_IDX = [
    6649, 8381, 8421, 8497, 8535, 8569, 8611,  # TOYOTA
    10472,                                        # NISSAN
    22789, 22896,                                 # BMW
    26444,                                        # VOLKSWAGEN
    32429,                                        # PEUGEOT
]

# Verify nothing unexpected before dropping
assert all(i in df.index for i in CONTAMINATED_IDX), "Index mismatch — re-check"
print("=== ROWS TO BE REMOVED ===")
print(df.loc[CONTAMINATED_IDX][["make","model","year","price"]].to_string())

# Remove and save
df_clean = df.drop(index=CONTAMINATED_IDX)
df_clean.to_csv("dev_lookup_filtered_evaluated.csv", index=False)

print(f"\nRemoved: {len(CONTAMINATED_IDX)} rows")
print(f"Remaining: {len(df_clean):,} rows")

=== ROWS TO BE REMOVED ===
             make                model    year     price
6649       TOYOTA   26HX DC 4X4P2.7LAT  2026.0  112770.0
8381       TOYOTA                GS300  2008.0   11115.0
8421       TOYOTA  26HX DC 4X4D2.4L AT  2026.0    9300.0
8497       TOYOTA  HX DC 4X4 D 2.4L AT  2026.0   96000.0
8535       TOYOTA  PARISIENNE BROUGHAM  1986.0    9000.0
8569       TOYOTA                  ES8  2024.0   31160.0
8611       TOYOTA                   H3  2006.0   73700.0
10472      NISSAN                   MT  2007.0    7100.0
22789         BMW                  CAR  2020.0  150000.0
22896         BMW                   IQ  2012.0   52500.0
26444  VOLKSWAGEN           JETTA 2.5L  2016.0   21271.0
32429     PEUGEOT      LAND REK - X MT  2025.0   63788.0

Removed: 12 rows
Remaining: 39,067 rows


In [45]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")

CONTAMINATED_IDX = [
    6649, 8381, 8421, 8497, 8535, 8569, 8611,  # TOYOTA
    10472,                                        # NISSAN
    22789, 22896,                                 # BMW
    26444,                                        # VOLKSWAGEN
    32429,                                        # PEUGEOT
]

# Verify nothing unexpected before dropping
assert all(i in df.index for i in CONTAMINATED_IDX), "Index mismatch — re-check"
print("=== ROWS TO BE REMOVED ===")
print(df.loc[CONTAMINATED_IDX][["make","model","year","price"]].to_string())

# Remove and save
df_clean = df.drop(index=CONTAMINATED_IDX)
df_clean.to_csv("dev_lookup_filtered_evaluated.csv", index=False)

print(f"\nRemoved: {len(CONTAMINATED_IDX)} rows")
print(f"Remaining: {len(df_clean):,} rows")

=== ROWS TO BE REMOVED ===
             make   model    year     price
6649       TOYOTA    RUSH  2023.0   51765.0
8381       TOYOTA   PRIUS  2012.0    5700.0
8421       TOYOTA  SIENNA  2014.0   28100.0
8497       TOYOTA   HIACE  2015.0   48400.0
8535       TOYOTA   HILUX  2014.0   21200.0
8569       TOYOTA   CAMRY  2001.0    4700.0
8611       TOYOTA   CAMRY  1998.0    3300.0
10472      NISSAN   URVAN  2016.0   31300.0
22789         BMW    840I  2024.0  220000.0
22896         BMW      X6  2013.0   40800.0
26444  VOLKSWAGEN  TIGUAN  2013.0   22400.0
32429     PEUGEOT  EXPERT  2020.0   34600.0

Removed: 12 rows
Remaining: 39,055 rows


In [46]:
# Identify by actual content — not index
contaminated_mask = (
    ((df["make"] == "TOYOTA") & (df["model"].isin(["GS300","PARISIENNE BROUGHAM","ES8","H3"])))
  | ((df["make"] == "TOYOTA") & df["model"].str.contains(r'\d+HX|HX DC', na=False, regex=True))
  | ((df["make"] == "BMW")    & (df["model"].isin(["CAR","IQ"])))
  | ((df["make"] == "NISSAN") & (df["model"] == "MT"))
  | ((df["make"] == "VOLKSWAGEN") & (df["model"] == "JETTA 2.5L"))
  | ((df["make"] == "PEUGEOT") & (df["model"] == "LAND REK - X MT"))
)

print("Rows matched by content:")
print(df[contaminated_mask][["make","model","year","price"]].to_string())
print(f"\nTotal: {contaminated_mask.sum()}")

Rows matched by content:
Empty DataFrame
Columns: [make, model, year, price]
Index: []

Total: 0


In [51]:
df = pd.read_csv('dev_lookup_filtered.csv')

In [52]:
df.shape

(39079, 9)

In [53]:
import pandas as pd

evaluated = pd.read_csv("dev_lookup_filtered_evaluated.csv")
filtered  = pd.read_csv("dev_lookup_filtered.csv")

print(f"filtered rows:  {len(filtered):,}   (expect 39,079)")
print(f"evaluated rows: {len(evaluated):,}  (expect 39,079)")
print(f"Row count match: {len(filtered) == len(evaluated)}")

# Spot-check the contaminated rows are still present (content-based)
contamination_check = (
    ((evaluated["make"] == "TOYOTA") & (evaluated["model"].isin(["GS300","PARISIENNE BROUGHAM","ES8","H3"])))
  | ((evaluated["make"] == "TOYOTA") & evaluated["model"].str.contains(r'\d+HX|HX DC', na=False, regex=True))
  | ((evaluated["make"] == "BMW")    & (evaluated["model"].isin(["CAR","IQ"])))
  | ((evaluated["make"] == "NISSAN") & (evaluated["model"] == "MT"))
  | ((evaluated["make"] == "VOLKSWAGEN") & (evaluated["model"] == "JETTA 2.5L"))
  | ((evaluated["make"] == "PEUGEOT") & (evaluated["model"] == "LAND REK - X MT"))
)
print(f"\nContaminated rows present: {contamination_check.sum()} (expect 12)")
print(evaluated[contamination_check][["make","model","year","price"]].to_string())

filtered rows:  39,079   (expect 39,079)
evaluated rows: 39,055  (expect 39,079)
Row count match: False

Contaminated rows present: 0 (expect 12)
Empty DataFrame
Columns: [make, model, year, price]
Index: []


In [54]:
import pandas as pd

filtered  = pd.read_csv("dev_lookup_filtered.csv")
evaluated = pd.read_csv("dev_lookup_filtered_evaluated.csv")

print(f"filtered:  {len(filtered):,}")
print(f"evaluated: {len(evaluated):,}")
print(f"missing:   {len(filtered) - len(evaluated):,}")

# Find which rows from filtered didn't make it into evaluated
# Use chassisNumber as the join key if it's reliable, else use positional index
if "chassisNumber" in filtered.columns and "chassisNumber" in evaluated.columns:
    filtered_ids  = set(filtered["chassisNumber"].dropna())
    evaluated_ids = set(evaluated["chassisNumber"].dropna())
    missing_ids   = filtered_ids - evaluated_ids
    print(f"\nMissing chassis IDs: {len(missing_ids)}")
    missing_rows = filtered[filtered["chassisNumber"].isin(missing_ids)]
    print(missing_rows[["make","model","year","trim","price"]].to_string())
else:
    print("No chassisNumber column — check join key")

# Also check: does evaluated have rows NOT in filtered? (shouldn't happen)
extra_ids = evaluated_ids - filtered_ids
print(f"\nRows in evaluated but not in filtered: {len(extra_ids)}")

filtered:  39,079
evaluated: 39,055
missing:   24

Missing chassis IDs: 24
             make                model    year        trim     price
6649       TOYOTA   26HX DC 4X4P2.7LAT  2026.0         GLX  112770.0
6650       TOYOTA                 RUSH  2023.0          GL   51765.0
8381       TOYOTA                GS300  2008.0         MID   11115.0
8383       TOYOTA                PRIUS  2012.0         EXI    5700.0
8421       TOYOTA  26HX DC 4X4D2.4L AT  2026.0         DLX    9300.0
8424       TOYOTA               SIENNA  2014.0          LE   28100.0
8497       TOYOTA  HX DC 4X4 D 2.4L AT  2026.0         DLX   96000.0
8501       TOYOTA                HIACE  2015.0          ST   48400.0
8535       TOYOTA  PARISIENNE BROUGHAM  1986.0         XLI    9000.0
8540       TOYOTA                HILUX  2014.0          DL   21200.0
8569       TOYOTA                  ES8  2024.0          SE   31160.0
8575       TOYOTA                CAMRY  2001.0         XLI    4700.0
8611       TOYOTA           

In [55]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")
assert len(df) == 39079, f"Expected 39,079 rows, got {len(df)}"

contaminated_mask = (
    ((df["make"] == "TOYOTA") & (df["model"].isin(["GS300","PARISIENNE BROUGHAM","ES8","H3"])))
  | ((df["make"] == "TOYOTA") & df["model"].str.contains(r'\d+HX|HX DC', na=False, regex=True))
  | ((df["make"] == "BMW")    & (df["model"].isin(["CAR","IQ"])))
  | ((df["make"] == "NISSAN") & (df["model"] == "MT"))
  | ((df["make"] == "VOLKSWAGEN") & (df["model"] == "JETTA 2.5L"))
  | ((df["make"] == "PEUGEOT") & (df["model"] == "LAND REK - X MT"))
)

print(f"Rows matched: {contaminated_mask.sum()} (expect 12)")
print(df[contaminated_mask][["make","model","year","price"]].to_string())

# Only save if the count and content are exactly right
if contaminated_mask.sum() == 12:
    df_clean = df[~contaminated_mask].copy()
    df_clean.to_csv("dev_lookup_filtered_evaluated.csv", index=False)
    print(f"\nSaved. Remaining: {len(df_clean):,} (expect 39,067)")
else:
    print("\nCount mismatch — not saved. Investigate before proceeding.")

Rows matched: 12 (expect 12)
             make                model    year     price
6649       TOYOTA   26HX DC 4X4P2.7LAT  2026.0  112770.0
8381       TOYOTA                GS300  2008.0   11115.0
8421       TOYOTA  26HX DC 4X4D2.4L AT  2026.0    9300.0
8497       TOYOTA  HX DC 4X4 D 2.4L AT  2026.0   96000.0
8535       TOYOTA  PARISIENNE BROUGHAM  1986.0    9000.0
8569       TOYOTA                  ES8  2024.0   31160.0
8611       TOYOTA                   H3  2006.0   73700.0
10472      NISSAN                   MT  2007.0    7100.0
22789         BMW                  CAR  2020.0  150000.0
22896         BMW                   IQ  2012.0   52500.0
26444  VOLKSWAGEN           JETTA 2.5L  2016.0   21271.0
32429     PEUGEOT      LAND REK - X MT  2025.0   63788.0

Saved. Remaining: 39,067 (expect 39,067)


In [56]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered_evaluated.csv")
assert len(df) == 39067, f"Unexpected row count: {len(df)}"

resolved   = df[df["predicted_price"].notna()].copy()
unresolved = df[df["predicted_price"].isna()].copy()

print(f"Resolved:   {len(resolved):,}  ({100*len(resolved)/len(df):.1f}%)")
print(f"Unresolved: {len(unresolved):,}  ({100*len(unresolved)/len(df):.1f}%)")

def metrics(subset, label):
    if len(subset) == 0:
        print(f"\n{label}: NO ROWS")
        return
    print(f"\n{'='*55}")
    print(f"{label}  (n={len(subset):,})")
    print(f"{'='*55}")
    print(f"  MAE         : {subset['abs_diff'].mean():>14,.0f} AED")
    print(f"  MAPE        : {subset['abs_diff_pct'].mean():>13.2f}%")
    print(f"  Median APE  : {subset['abs_diff_pct'].median():>13.2f}%")
    print(f"  p90 APE     : {subset['abs_diff_pct'].quantile(.90):>13.2f}%")
    print(f"  p95 APE     : {subset['abs_diff_pct'].quantile(.95):>13.2f}%")
    print(f"  Avg sim     : {subset['avg_similarity'].mean():>13.3f}")
    print(f"  Avg neighbors:{subset['n_neighbors'].mean():>12.1f}")

metrics(resolved, "ALL RESOLVED")
for tier, grp in resolved.groupby("provenance"):
    metrics(grp, f"provenance={tier}")

Resolved:   38,480  (98.5%)
Unresolved: 587  (1.5%)

ALL RESOLVED  (n=38,480)
  MAE         :         17,273 AED
  MAPE        :         27.47%
  Median APE  :         13.12%
  p90 APE     :         47.85%
  p95 APE     :         77.35%
  Avg sim     :         0.907
  Avg neighbors:         9.5

provenance=exact  (n=38,248)
  MAE         :         17,183 AED
  MAPE        :         26.85%
  Median APE  :         13.06%
  p90 APE     :         47.17%
  p95 APE     :         75.67%
  Avg sim     :         0.908
  Avg neighbors:         9.5

provenance=make_fallback  (n=232)
  MAE         :         31,978 AED
  MAPE        :        129.15%
  Median APE  :         41.41%
  p90 APE     :        208.29%
  p95 APE     :        412.88%
  Avg sim     :         0.731
  Avg neighbors:         9.8


In [57]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered.csv")

# Year-conditional floors to evaluate
year_floors = [
    (2020, 20_000),
    (2015, 10_000),
    (2010,  5_000),
    (2000,  2_000),
]

print("=== YEAR-CONDITIONAL FLOOR IMPACT ===")
print(f"Total rows: {len(df):,}\n")

would_flag = pd.Series(False, index=df.index)

for year_thresh, floor in year_floors:
    mask = (df["year"] >= year_thresh) & (df["price"] < floor) & ~would_flag
    would_flag |= mask
    print(f"year >= {year_thresh} AND price < {floor:,} AED: {mask.sum()} rows")
    if mask.sum() > 0:
        print(df[mask][["make","model","year","trim","price"]]
              .sort_values("price").head(20).to_string())
    print()

print(f"Total new flags from year-conditional floor: {would_flag.sum()}")

# Cross-check: how many of these are already flagged in the filtered file?
if "price_flag" in df.columns:
    already_flagged = df[would_flag & df["price_flag"]]
    print(f"Already flagged in filtered file: {len(already_flagged)}")
    print(f"Net new flags: {would_flag.sum() - len(already_flagged)}")

=== YEAR-CONDITIONAL FLOOR IMPACT ===
Total rows: 39,079

year >= 2020 AND price < 20,000 AED: 135 rows
             make     model    year         trim   price
38906     MAEXTRO      S800  2025.0            S   950.0
38970      BAOJUN       NaN  2024.0          STD  1000.0
38910         VGV  U75 PLUS  2023.0          NaN  2000.0
38524        TATA   LPO1618  2026.0            S  2150.0
26051      SUZUKI     SWIFT  2023.0           GL  3900.0
8329       TOYOTA    PREVIA  2027.0          STD  4265.0
13231  MITSUBISHI    PAJERO  2027.0  GLS MIDLINE  4305.0
14032       HONDA       FIT  2022.0           GL  4465.0
13861       HONDA      HR-V  2020.0          NaN  5135.0
36481     VICTORY        V1  2022.0          STD  5600.0
26679  VOLKSWAGEN      TERA  2024.0          STD  6100.0
16044         KIA        K5  2020.0           LE  6365.0
24044         BMW       NaN  2020.0           LE  6365.0
8584       TOYOTA    ALLION  2021.0           LE  6365.0
25971      SUZUKI     DZIRE  2023.0      

In [58]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered.csv")

# Rule 1 — sentinel prices
sentinel_prices = [6_365, 1_330, 6_400]
r1 = df["price"].isin(sentinel_prices)
print(f"Rule 1 — sentinel prices: {r1.sum()} rows")
print(df[r1]["price"].value_counts().to_string())

# Rule 2 — impossible future years
r2 = df["year"] > 2026
print(f"\nRule 2 — year > 2026: {r2.sum()} rows")
print(df[r2][["make","model","year","price"]].to_string())

# Rule 3 — absolute floor
r3 = df["price"] < 1_500
print(f"\nRule 3 — price < 1,500 AED: {r3.sum()} rows")
print(df[r3][["make","model","year","trim","price"]]
      .sort_values("price").to_string())

# Combined
any_rule = r1 | r2 | r3
print(f"\nTotal flagged by all three rules: {any_rule.sum()}")
print(f"Overlap (already price_flagged): {df[any_rule & df['price_flag']].shape[0]}")


Rule 1 — sentinel prices: 89 rows
price
6400.0    51
6365.0    33
1330.0     5

Rule 2 — year > 2026: 18 rows
             make       model    year     price
5031         BAIC     U5 PLUS  2027.0   61740.0
8329       TOYOTA      PREVIA  2027.0    4265.0
13231  MITSUBISHI      PAJERO  2027.0    4305.0
29638         BYD       SHARK  2027.0  158900.0
33263       GEELY         EX5  2027.0   90000.0
33267       GEELY         EX5  2027.0  114900.0
33361       GEELY       CRUZE  2027.0   17195.0
33365       GEELY     COOLRAY  2027.0   68400.0
33367       GEELY     EMGRAND  2027.0   44500.0
34112         GAC  GS3 EMZOOM  2027.0   72000.0
34903       HAVAL          V7  2027.0  138750.0
36035        BAIC     U5 PLUS  2027.0   39000.0
36048        BAIC    BJ40 PRO  2027.0  136400.0
36063        BAIC    BJ40 PRO  2027.0  114300.0
36078        BAIC    BJ40 PRO  2027.0  170000.0
36083        BAIC    BJ40 PRO  2027.0  175000.0
38412         BYD          B5  2027.0  141100.0
38955         TAM         

In [59]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered.csv")
assert len(df) == 39079, f"Unexpected row count: {len(df)}"

SENTINEL_PRICES = [6_365.0, 6_400.0, 1_330.0]
HARD_FLOOR      = 1_500.0

r1 = df["price"].isin(SENTINEL_PRICES)
r3 = df["price"] < HARD_FLOOR
new_flag = r1 | r3

print(f"Rule 1 (sentinel):  {r1.sum():,}")
print(f"Rule 3 (hard floor): {r3.sum():,}")
print(f"Combined (unique):  {new_flag.sum():,}")

# Add to existing price_flag column
df.loc[new_flag, "price_flag"]   = True
df.loc[new_flag & (df["flag_reason"] == ""), "flag_reason"] = "sentinel_or_floor"
df.loc[r1 & (df["flag_reason"] == "sentinel_or_floor"), "flag_reason"] = "sentinel_price"
df.loc[r3 & ~r1 & (df["flag_reason"] == ""), "flag_reason"] = "below_hard_floor_1500"

print(f"\nTotal price_flag=True after update: {df['price_flag'].sum():,}")
print(f"flag_reason value counts:\n{df['flag_reason'].value_counts().to_string()}")

# Sanity check — make sure we didn't touch legitimate rows
print("\n=== SAMPLE OF NEWLY FLAGGED ROWS ===")
newly_flagged = df[new_flag]
print(newly_flagged[["make","model","year","trim","price","flag_reason"]]
      .sort_values("price").head(20).to_string())

# Save only after visual confirmation
# df.to_csv("dev_lookup_filtered.csv", index=False)

Rule 1 (sentinel):  89
Rule 3 (hard floor): 37
Combined (unique):  121

Total price_flag=True after update: 121
flag_reason value counts:
flag_reason
insufficient_data_no_flag    158

=== SAMPLE OF NEWLY FLAGGED ROWS ===
             make       model    year  trim    price                flag_reason
5874       TOYOTA       CAMRY  2000.0    CE   650.00                        NaN
10601      NISSAN  PATHFINDER  1999.0    XE   760.00                        NaN
8746       NISSAN       SUNNY  1997.0    SL   800.00                        NaN
8806       NISSAN       SUNNY  1997.0   STD   800.00                        NaN
9867       NISSAN  PATHFINDER  2000.0    XE   800.00                        NaN
10327      NISSAN       SUNNY  1997.0     S   800.00                        NaN
38974     PONTIAC    TRANS AM  1990.0     G   900.00  insufficient_data_no_flag
38906     MAEXTRO        S800  2025.0     S   950.00  insufficient_data_no_flag
34571    DAIHATSU         NaN  2009.0   NaN  1000.00       

In [60]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered.csv")
assert len(df) == 39079

# Diagnose flag_reason dtype before touching anything
print("flag_reason dtype:", df["flag_reason"].dtype)
print("Null count:", df["flag_reason"].isna().sum())
print("Empty string count:", (df["flag_reason"] == "").sum())
print("Sample values:", df["flag_reason"].unique()[:10])

SENTINEL_PRICES = [6_365.0, 6_400.0, 1_330.0]
HARD_FLOOR      = 1_500.0

r1 = df["price"].isin(SENTINEL_PRICES)
r3 = df["price"] < HARD_FLOOR
new_flag = r1 | r3

# Normalize flag_reason to empty string before writing
df["flag_reason"] = df["flag_reason"].fillna("")
df["price_flag"]  = df["price_flag"].fillna(False)

# Apply flags — sentinel takes precedence over floor
df.loc[r1, "price_flag"]   = True
df.loc[r1, "flag_reason"]  = "sentinel_price"
df.loc[r3 & ~r1, "price_flag"]  = True
df.loc[r3 & ~r1, "flag_reason"] = "below_hard_floor_1500"

print(f"\nTotal price_flag=True: {df['price_flag'].sum():,}  (expect 121)")
print(f"\nflag_reason value counts:")
print(df["flag_reason"].value_counts().to_string())

print("\n=== SAMPLE NEWLY FLAGGED — SENTINELS ===")
print(df[r1][["make","model","year","price","flag_reason"]].head(10).to_string())

print("\n=== SAMPLE NEWLY FLAGGED — FLOOR ===")
print(df[r3 & ~r1][["make","model","year","price","flag_reason"]].head(10).to_string())

# Save only after output confirmed
# df.to_csv("dev_lookup_filtered.csv", index=False)

flag_reason dtype: str
Null count: 38921
Empty string count: 0
Sample values: <ArrowStringArray>
[nan, 'insufficient_data_no_flag']
Length: 2, dtype: str

Total price_flag=True: 121  (expect 121)

flag_reason value counts:
flag_reason
                             38803
insufficient_data_no_flag      155
sentinel_price                  89
below_hard_floor_1500           32

=== SAMPLE NEWLY FLAGGED — SENTINELS ===
          make     model    year   price     flag_reason
103     TOYOTA     CAMRY  2002.0  6365.0  sentinel_price
527      HONDA     CIVIC  2006.0  6400.0  sentinel_price
667     NISSAN     SUNNY  2009.0  6365.0  sentinel_price
1619      MINI    COOPER  2003.0  6365.0  sentinel_price
1948  CADILLAC  ESCALADE  2002.0  6365.0  sentinel_price
4006     MAZDA         2  2011.0  6365.0  sentinel_price
4025      FORD      FIGO  2012.0  6365.0  sentinel_price
4946    TOYOTA        XA  2006.0  6400.0  sentinel_price
5716    TOYOTA     CAMRY  2000.0  1330.0  sentinel_price
6091    TOYOT

In [61]:
df.to_csv("dev_lookup_filtered.csv", index=False)
print("Saved. Verify:")
df2 = pd.read_csv("dev_lookup_filtered.csv")
print(f"  rows: {len(df2):,}  (expect 39,079)")
print(f"  price_flag=True: {df2['price_flag'].sum():,}  (expect 121)")
print(f"  flag_reason counts:\n{df2['flag_reason'].value_counts().to_string()}")

Saved. Verify:
  rows: 39,079  (expect 39,079)
  price_flag=True: 121  (expect 121)
  flag_reason counts:
flag_reason
insufficient_data_no_flag    155
sentinel_price                89
below_hard_floor_1500         32


In [62]:
df = pd.read_csv("dev_lookup_filtered.csv")
clean_pool = df[~df["price_flag"]].copy()  # exclude flagged rows from neighbor pool
# then run LOO against clean_pool, not df

In [63]:
clean_pool.to_csv("clean_pool.csv",index=False)

In [64]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered.csv")
clean = df[~df["price_flag"]].copy()
print(f"Clean pool: {len(clean):,} rows")

# Year-conditional floors — conservative, model-aware
year_floors = [
    (2020, 15_000),
    (2015,  8_000),
    (2010,  4_000),
]

print("\n=== YEAR-CONDITIONAL FLOOR CANDIDATES ===")
already_new_flag = pd.Series(False, index=clean.index)

for year_thresh, floor in year_floors:
    mask = (clean["year"] >= year_thresh) & (clean["price"] < floor) & ~already_new_flag
    already_new_flag |= mask
    print(f"\nyear >= {year_thresh} AND price < {floor:,} AED: {mask.sum()} rows")
    print(clean[mask][["make","model","year","trim","price"]]
          .sort_values("price").to_string())

Clean pool: 38,958 rows

=== YEAR-CONDITIONAL FLOOR CANDIDATES ===

year >= 2020 AND price < 15,000 AED: 50 rows
                make                model    year                 trim    price
38910            VGV             U75 PLUS  2023.0                  NaN   2000.0
38524           TATA              LPO1618  2026.0                    S   2150.0
26051         SUZUKI                SWIFT  2023.0                   GL   3900.0
8329          TOYOTA               PREVIA  2027.0                  STD   4265.0
13231     MITSUBISHI               PAJERO  2027.0          GLS MIDLINE   4305.0
14032          HONDA                  FIT  2022.0                   GL   4465.0
13861          HONDA                 HR-V  2020.0                  NaN   5135.0
36481        VICTORY                   V1  2022.0                  STD   5600.0
26679     VOLKSWAGEN                 TERA  2024.0                  STD   6100.0
10002         NISSAN               ALTIMA  2022.0                  XLI   6700.0
36487  

In [65]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered.csv")
clean = df[~df["price_flag"]].copy()

LUXURY_MAKES = {
    "MERCEDES BENZ", "BMW", "AUDI", "LEXUS", "PORSCHE",
    "CADILLAC", "LINCOLN", "INFINITI", "LAND ROVER",
    "BENTLEY", "FERRARI", "LAMBORGHINI", "MASERATI",
    "ROLLS ROYCE", "ASTON MARTIN", "DODGE", "JEEP"
}

# Rule A — luxury make, year >= 2015, price < 15,000
r_luxury = (
    (clean["year"] >= 2015) &
    (clean["make"].isin(LUXURY_MAKES)) &
    (clean["price"] < 15_000)
)

# Rule B — any make, year >= 2010, price < 4,000
r_floor_2010 = (
    (clean["year"] >= 2010) &
    (clean["price"] < 4_000)
)

new_flag = r_luxury | r_floor_2010

print(f"Rule A (luxury + year>=2015 + price<15K): {r_luxury.sum()} rows")
print(clean[r_luxury][["make","model","year","trim","price"]]
      .sort_values("price").to_string())

print(f"\nRule B (year>=2010 + price<4K): {r_floor_2010.sum()} rows")
print(clean[r_floor_2010][["make","model","year","trim","price"]]
      .sort_values("price").to_string())

print(f"\nTotal new flags (unique): {new_flag.sum()}")
print(f"Overlap between A and B: {(r_luxury & r_floor_2010).sum()}")

Rule A (luxury + year>=2015 + price<15K): 32 rows
                make          model    year            trim    price
29928          DODGE        DURANGO  2016.0         LIMITED   6650.0
22726            BMW             X5  2016.0             STD   7140.0
33698        LINCOLN            MKT  2018.0             GLX   7220.0
28714       INFINITI           QX80  2016.0             TOP   7300.0
11340          LEXUS          IS250  2015.0         PREMIER   9120.0
32654       CADILLAC          LYRIQ  2024.0  LAUNCH EDITION   9200.0
11737          LEXUS          IS300  2019.0               V   9310.0
33769        LINCOLN             L6  2024.0            BASE   9690.0
28563       INFINITI            Q50  2015.0         PREMIUM  10000.0
28873       INFINITI           QX80  2016.0      EXCELLENCE  10070.0
12075          LEXUS          RX270  2015.0         PREMIER  10200.0
29775          DODGE        JOURNEY  2015.0             STD  10710.0
16819  MERCEDES BENZ           S400  2015.0          

In [66]:
import pandas as pd

df = pd.read_csv("dev_lookup_filtered.csv")
assert len(df) == 39079

LUXURY_MAKES = {
    "MERCEDES BENZ", "BMW", "AUDI", "LEXUS", "PORSCHE",
    "CADILLAC", "LINCOLN", "INFINITI", "LAND ROVER",
    "BENTLEY", "FERRARI", "LAMBORGHINI", "MASERATI",
    "ROLLS ROYCE", "ASTON MARTIN", "DODGE", "JEEP"
}

r_luxury = (
    (df["year"] >= 2015) &
    (df["make"].isin(LUXURY_MAKES)) &
    (df["price"] < 15_000) &
    (~df["price_flag"])
)

r_floor_2010 = (
    (df["year"] >= 2010) &
    (df["price"] < 4_000) &
    (~df["price_flag"])
)

new_flag = r_luxury | r_floor_2010

# Verify count before touching anything
print(f"Rule A: {r_luxury.sum()} rows")
print(f"Rule B: {r_floor_2010.sum()} rows")
print(f"Combined: {new_flag.sum()} rows (expect 45)")
assert new_flag.sum() == 45, "Count mismatch — re-check"

# Apply
df["flag_reason"] = df["flag_reason"].fillna("")
df.loc[r_luxury, "price_flag"]  = True
df.loc[r_luxury, "flag_reason"] = "luxury_price_floor"
df.loc[r_floor_2010 & ~r_luxury, "price_flag"]  = True
df.loc[r_floor_2010 & ~r_luxury, "flag_reason"] = "year_price_floor_2010"

print(f"\nTotal price_flag=True: {df['price_flag'].sum():,}  (expect 166)")
print(f"\nflag_reason counts:\n{df['flag_reason'].value_counts().to_string()}")

# Save only after confirmation
# df.to_csv("dev_lookup_filtered.csv", index=False)

Rule A: 32 rows
Rule B: 13 rows
Combined: 45 rows (expect 45)

Total price_flag=True: 166  (expect 166)

flag_reason counts:
flag_reason
                             38760
insufficient_data_no_flag      153
sentinel_price                  89
below_hard_floor_1500           32
luxury_price_floor              32
year_price_floor_2010           13


In [67]:
df.to_csv("dev_lookup_filtered.csv", index=False)
print("Saved.")
df2 = pd.read_csv("dev_lookup_filtered.csv")
print(f"  rows: {len(df2):,}  (expect 39,079)")
print(f"  price_flag=True: {df2['price_flag'].sum():,}  (expect 166)")

Saved.
  rows: 39,079  (expect 39,079)
  price_flag=True: 166  (expect 166)


In [68]:
for m in unresolved_rows[unresolved_rows["make"]=="MERCEDES BENZ"]["model"].head(20):
    count = (cleaned_deduped[(cleaned_deduped["make"]=="MERCEDES BENZ") & (cleaned_deduped["model"]==m)]).shape[0]
    print(f"{m!r}: {count} rows in dataset with this exact model")

'CLK230 K': 2 rows in dataset with this exact model
'B150': 1 rows in dataset with this exact model
'EQG G580': 1 rows in dataset with this exact model
'V280': 1 rows in dataset with this exact model
'ML430': 1 rows in dataset with this exact model
'V220 D': 1 rows in dataset with this exact model
'G55': 1 rows in dataset with this exact model
'S400H': 1 rows in dataset with this exact model
'GLE580': 1 rows in dataset with this exact model
'CLA180': 1 rows in dataset with this exact model
'G650 MAYBACH': 1 rows in dataset with this exact model
'CLC180K': 1 rows in dataset with this exact model
'SL43': 1 rows in dataset with this exact model
'GLA43 AMG': 1 rows in dataset with this exact model
'EQS 53 AMG': 1 rows in dataset with this exact model
'A45 AMG': 1 rows in dataset with this exact model
'500 SEL': 1 rows in dataset with this exact model
'ML550': 1 rows in dataset with this exact model
'B250 E': 1 rows in dataset with this exact model
'CLE450': 1 rows in dataset with this exac

In [12]:
import pandas as pd

In [13]:
df = pd.read_csv("no_dupes.csv")

In [14]:
dupes = df.duplicated(subset=['chassisNumber']).sum()


In [15]:
dupes

np.int64(0)

In [9]:
df = df.drop_duplicates(subset=['chassisNumber'], keep='first')

In [16]:
df.shape

(995034, 12)

In [11]:
df.to_csv("no_dupes.csv",index=False)

In [17]:
df.isna().sum()

chassisNumber         0
year                 10
make                 10
model              5915
trim              21385
bodyType              1
regionalSpec        222
cylinders         17433
color             63569
origin               48
weightInKg         1818
noOfPassengers        0
dtype: int64

In [21]:
df.duplicated(subset=['chassisNumber']).sum()

np.int64(0)

In [3]:
import pandas as pd
df = pd.read_csv("dev_lookup_filtered.csv")

print(df["year"].isna().sum())

print(df[df["year"].isna()][
    ["chassisNumber", "make", "model", "trim", "price"]
])

3
           chassisNumber     make      model                       trim  \
8195   JTDZ13FV8RD126072   TOYOTA       RAV4                         EX   
29499  LCOCF4CD3R0454375      BYD       TANG  EV CHAMPION EDITION 600KM   
33789  SLMJJ2LT8JEL14267  LINCOLN  NAVIGATOR                    RESERVE   

         price  
8195   80500.0  
29499  80800.0  
33789  95000.0  


In [4]:
df = pd.read_csv("dev_lookup_filtered.csv")

print("Missing years:", df["year"].isna().sum())

Missing years: 3


In [1]:
import pandas as pd

low = pd.read_csv("reports/low_variance_groups.csv")
high = pd.read_csv("reports/high_variance_groups.csv")

cols = [
    "make",
    "model",
    "year",
    "trim",
    "count",
    "median_price",
    "iqr_pct",
    "cv_pct",
]

print("\n===== LOW VARIANCE =====")
print(low[cols].head(20))

print("\n===== HIGH VARIANCE =====")
print(high[cols].head(20))


===== LOW VARIANCE =====
             make        model    year        trim  count  median_price  \
0      MITSUBISHI  FUSO CANTER  2020.0         STD      5       72800.0   
1           LEXUS      ES300 H  2020.0     PREMIER      5       68100.0   
2             GMC        YUKON  2025.0         SLE      5      240000.0   
3      MITSUBISHI       CANTER  2014.0         STD      5       44000.0   
4       CHEVROLET       GROOVE  2024.0          LT      5       44700.0   
5         HYUNDAI       ACCENT  2020.0          GL      6       26600.0   
6          TOYOTA        RAIZE  2023.0           E      5       47500.0   
7           LEXUS        LS430  2003.0         TOP      5       10500.0   
8             KIA   SPORTAGE L  2025.0          LX      5       80000.0   
9          NISSAN        SUNNY  2021.0          SV      5       26400.0   
10  MERCEDES BENZ         E200  2017.0         STD      5       75500.0   
11            KIA      SORENTO  2022.0          EX      5       93000.0   

In [1]:
import pandas as pd

df = pd.read_csv("experiment001/experiment001_predictions.csv")

print(df.columns.tolist())

['chassisNumber', 'year', 'make', 'model', 'trim', 'price', 'price_flag', 'flag_reason', 'chassis_unreliable', 'predicted_price', 'price_diff', 'abs_diff', 'price_diff_pct', 'abs_diff_pct']


In [2]:
import pandas as pd

df = pd.read_csv("experiment001/experiment001_predictions.csv")

ape = df["abs_diff_pct"]
ae = df["abs_diff"]

print("=" * 60)
print("CURRENT EXPERIMENT 001 METRICS")
print("=" * 60)

print(f"Rows        : {len(df):,}")
print(f"MAE         : {ae.mean():,.2f}")
print(f"Median AE   : {ae.median():,.2f}")
print(f"MAPE        : {ape.mean():.2f}%")
print(f"Median APE  : {ape.median():.2f}%")
print(f"Within 5%   : {(ape <= 5).mean()*100:.2f}%")
print(f"Within 10%  : {(ape <= 10).mean()*100:.2f}%")
print(f"Within 20%  : {(ape <= 20).mean()*100:.2f}%")

CURRENT EXPERIMENT 001 METRICS
Rows        : 7,768
MAE         : 33,513.34
Median AE   : 9,125.94
MAPE        : 34.99%
Median APE  : 21.29%
Within 5%   : 12.09%
Within 10%  : 25.49%
Within 20%  : 47.76%


In [3]:
import pandas as pd

df = pd.read_csv("experiment001/experiment001_predictions.csv")

print(df.shape)

print(df.head())

print(df["year"].value_counts().sort_index().tail())

(7768, 14)
  chassisNumber    year    make   model             trim     price  \
0           NaN  2025.0  NISSAN  PATROL           PRO 4X  191900.0   
1           NaN  2016.0  NISSAN  PATROL  SUPER SAFARI T1   55300.0   
2           NaN  2015.0  NISSAN  PATROL    SE T2 LEATHER   49400.0   
3           NaN  2009.0  NISSAN  PATROL        SAFARI T1   36700.0   
4           NaN  2016.0  NISSAN  PATROL    SE T2 LEATHER   61110.0   

   price_flag flag_reason  chassis_unreliable  predicted_price    price_diff  \
0       False         NaN               False    232217.087267  40317.087267   
1       False         NaN               False     94965.170108  39665.170108   
2       False         NaN               False     68024.755508  18624.755508   
3       False         NaN               False     38229.388434   1529.388434   
4       False         NaN               False     76928.005711  15818.005711   

       abs_diff  price_diff_pct  abs_diff_pct  
0  40317.087267       21.009425     21.

In [4]:
import pandas as pd

df = pd.read_csv("experiment001/experiment001_predictions.csv")

print(len(df))

7768


In [1]:
import pandas as pd

df = pd.read_csv("experiment001/experiment001_predictions.csv")
print(len(df))

7710
